# 02c: Broadcast, Game Time & Final Feature Assembly
Two things this notebook fixes before modeling:
1. `is_national`, `is_premium_national`, `is_day_game`, and the date-derived features (day of week, month, weekend) were only ever computed *inside* `03_eda.ipynb`'s memory, never saved back to `mets_master.csv`. This persists them.
2. Builds a single combined `mets_2026_master.csv` (games + weather + promotions + standings + starter ERA + broadcast + game time) for the completed 2026 games, matching the training schema exactly, so the model can be backtested against real 2026 results.

In [1]:
import pandas as pd
import numpy as np
import requests
import time

TEAM_ID = 121
SEASONS = [2022, 2023, 2024, 2025, 2026]

df = pd.read_csv("../data/mets_master.csv")
df_2026 = pd.read_csv("../data/mets_2026_standings_pitchers.csv")
print(f"Training master: {len(df)} rows")
print(f"2026 completed games: {len(df_2026)} rows")

Training master: 320 rows
2026 completed games: 59 rows


## National Broadcasts (2022-2026)

In [2]:
def get_national_broadcasts(team_id, seasons):
    national = {}
    national_networks = {}
    for season in seasons:
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {"sportId": 1, "season": season, "gameType": "R", "teamId": team_id, "hydrate": "broadcasts"}
        data = {}
        for attempt in range(3):
            try:
                data = requests.get(url, params=params, timeout=15).json()
                break
            except requests.exceptions.RequestException:
                if attempt == 2:
                    data = {}
                else:
                    time.sleep(2)
        for date in data.get("dates", []):
            for game in date.get("games", []):
                if game["teams"]["home"]["team"]["id"] != team_id:
                    continue
                pk = game["gamePk"]
                tv_national = [b["name"] for b in game.get("broadcasts", [])
                               if b.get("type") == "TV" and b.get("isNational")]
                national[pk] = 1 if tv_national else 0
                national_networks[pk] = ", ".join(tv_national) if tv_national else None
        print(f"  {season}: broadcast data pulled")
    return national, national_networks

print("Pulling national broadcast data...")
nat_flags, nat_networks = get_national_broadcasts(TEAM_ID, SEASONS)
print(f"Total games with broadcast data: {len(nat_flags)}")

premium_networks = {"FOX", "FOX, FOX", "ESPN/ESPN App", "ESPN/ESPN App, ESPN/ESPN App",
                    "Apple TV", "Apple TV, Apple TV", "TBS (out-of-market only)",
                    "TBS (out-of-market only), TBS (out-of-market only)",
                    "TBS (out-of-market only), TBS", "Roku", "Roku, Roku",
                    "FS1", "FS1, FS1"}

def apply_broadcast_features(frame):
    frame = frame.copy()
    frame["is_national"] = frame["game_pk"].map(nat_flags).fillna(0).astype(int)
    frame["national_network"] = frame["game_pk"].map(nat_networks)
    frame["is_premium_national"] = frame["national_network"].isin(premium_networks).astype(int)
    tier_map = {(0, 0): "Non-national", (1, 0): "MLBN only", (1, 1): "Premium national"}
    frame["broadcast_tier"] = list(zip(frame["is_national"], frame["is_premium_national"]))
    frame["broadcast_tier"] = frame["broadcast_tier"].map(tier_map)
    return frame

df = apply_broadcast_features(df)
df_2026 = apply_broadcast_features(df_2026)
print(df["broadcast_tier"].value_counts())

Pulling national broadcast data...
  2022: broadcast data pulled
  2023: broadcast data pulled
  2024: broadcast data pulled
  2025: broadcast data pulled
  2026: broadcast data pulled
Total games with broadcast data: 405
broadcast_tier
Non-national        185
MLBN only           103
Premium national     32
Name: count, dtype: int64


## Game Time: Day vs. Night (2022-2026)

In [3]:
def get_game_times(team_id, seasons):
    times = {}
    for season in seasons:
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {"sportId": 1, "season": season, "gameType": "R", "teamId": team_id,
                  "hydrate": "linescore,team"}
        data = {}
        for attempt in range(3):
            try:
                data = requests.get(url, params=params, timeout=15).json()
                break
            except requests.exceptions.RequestException:
                if attempt == 2:
                    data = {}
                else:
                    time.sleep(2)
        for date in data.get("dates", []):
            for game in date.get("games", []):
                if game["teams"]["home"]["team"]["id"] != team_id:
                    continue
                game_dt_str = game.get("gameDate", "")
                if game_dt_str:
                    dt = pd.to_datetime(game_dt_str, utc=True)
                    hour_et = (dt.hour - 4) % 24
                    minute_et = dt.minute
                    times[game["gamePk"]] = hour_et + minute_et / 60
        print(f"  {season}: game times pulled")
    return times

print("Pulling game start times...")
game_times = get_game_times(TEAM_ID, SEASONS)
print(f"Total games with time data: {len(game_times)}")

def time_bucket(hour):
    if pd.isna(hour):
        return None
    if hour < 17:
        return "Early (<5pm)"
    elif hour < 18.5:
        return "Twilight (5-6:30pm)"
    return "Night (6:30pm+)"

def apply_gametime_features(frame):
    frame = frame.copy()
    frame["game_hour_et"] = frame["game_pk"].map(game_times)
    frame["is_day_game"] = (frame["game_hour_et"] < 17).astype("Int64")
    frame["time_bucket"] = frame["game_hour_et"].apply(time_bucket)
    return frame

df = apply_gametime_features(df)
df_2026 = apply_gametime_features(df_2026)
print(df["time_bucket"].value_counts())

Pulling game start times...
  2022: game times pulled
  2023: game times pulled
  2024: game times pulled
  2025: game times pulled
  2026: game times pulled
Total games with time data: 405
time_bucket
Night (6:30pm+)        186
Early (<5pm)           133
Twilight (5-6:30pm)      1
Name: count, dtype: int64


## Date-Derived Features (day of week, month, weekend)

In [4]:
def apply_date_features(frame):
    frame = frame.copy()
    dt = pd.to_datetime(frame["date"])
    frame["day_of_week"] = dt.dt.day_name()
    frame["month"] = dt.dt.month
    frame["dayofweek"] = dt.dt.dayofweek
    frame["is_weekend"] = frame["dayofweek"].isin([4, 5, 6]).astype(int)
    return frame

df = apply_date_features(df)
df_2026 = apply_date_features(df_2026)
df.to_csv("../data/mets_master.csv", index=False)
print(f"Saved mets_master.csv with {df.shape[1]} columns, {len(df)} rows")

Saved mets_master.csv with 44 columns, 320 rows


## Assemble the Combined 2026 Master (weather + promotions)
`df_2026` already has games, standings, and starter ERA. This adds weather and promotions to match the training schema exactly.

In [5]:
weather_2026 = pd.read_csv("../data/mets_weather_2026.csv")
df_2026 = df_2026.merge(weather_2026, on="date", how="left")

promotions = pd.read_csv("../data/mets_promotions.csv")
promo_by_date = promotions.groupby("date").agg(
    n_promotions=("promotion", "count"),
    is_bobblehead=("type", lambda s: int((s == "bobblehead").any())),
    promotion_names=("promotion", lambda s: "; ".join(s)),
).reset_index()

df_2026 = df_2026.merge(promo_by_date, on="date", how="left")
df_2026["n_promotions"] = df_2026["n_promotions"].fillna(0).astype(int)
df_2026["is_bobblehead"] = df_2026["is_bobblehead"].fillna(0).astype(int)
df_2026["is_promo"] = (df_2026["n_promotions"] > 0).astype(int)

# Match training column order/set exactly
master_cols = list(df.columns)
missing_cols = [c for c in master_cols if c not in df_2026.columns]
print(f"Columns present in training master but missing here: {missing_cols}")
df_2026 = df_2026.reindex(columns=master_cols)

df_2026.to_csv("../data/mets_2026_master.csv", index=False)
print(f"Saved mets_2026_master.csv with {df_2026.shape[1]} columns, {len(df_2026)} rows")
print(f"Promo games: {df_2026['is_promo'].sum()} / {len(df_2026)}")
print(f"Attendance range: {df_2026['attendance'].min():.0f} - {df_2026['attendance'].max():.0f}")
df_2026.head()

Columns present in training master but missing here: []
Saved mets_2026_master.csv with 44 columns, 59 rows
Promo games: 14 / 59
Attendance range: 22672 - 41449


,date,season,opponent,home_score,away_score,attendance,status,game_pk,is_game2,avg_temp_f,...,national_network,is_premium_national,broadcast_tier,game_hour_et,is_day_game,time_bucket,day_of_week,month,dayofweek,is_weekend
0,2026-03-26,2026,Pittsburgh Pirates,11.0,7.0,41449.0,Final,823649,0,56.7,...,"NBC/Peacock, NBC/Peacock",0,MLBN only,13.250000,1,Early (<5pm),Thursday,3,3,0
1,2026-03-28,2026,Pittsburgh Pirates,4.0,2.0,37183.0,Final,823651,0,40.1,...,NaN,0,Non-national,16.166667,1,Early (<5pm),Saturday,3,5,1
2,2026-03-29,2026,Pittsburgh Pirates,3.0,4.0,36940.0,Final,823647,0,45.2,...,NaN,0,Non-national,13.666667,1,Early (<5pm),Sunday,3,6,1
3,2026-04-07,2026,Arizona Diamondbacks,4.0,3.0,34753.0,Final,823648,0,44.5,...,NaN,0,Non-national,16.166667,1,Early (<5pm),Tuesday,4,1,0
4,2026-04-08,2026,Arizona Diamondbacks,2.0,7.0,33422.0,Final,823646,0,41.7,...,NaN,0,Non-national,16.166667,1,Early (<5pm),Wednesday,4,2,0
